In [ ]:
import pandas as pd
from slrrag.settings import PATH_CSV_03_AGENTS_ANWSERS
from slrrag.input import FIRST_ROUND_QUESTIONS
from slrrag.toolbox.toolbox_notebooks import plot_histogram

df_analyisis = pd.read_csv(PATH_CSV_03_AGENTS_ANWSERS)
df_analyisis

In [ ]:
df_analyisis[df_analyisis["is_reference"]]

In [ ]:
from slrrag.input import OLLAMA_MODELS


type_list_col = {
    q + "_" + x: "bool" for x in OLLAMA_MODELS for q in FIRST_ROUND_QUESTIONS.keys()
}
type_list_col

In [ ]:
kept_columns = [
    x
    for x in type_list_col.keys()
    if x.startswith(("risk_nautical", "nautical_offshore", "probability_collision"))
]
df_seems_relevant = df_analyisis[df_analyisis[kept_columns].any(axis=1)]
df_seems_relevant

In [ ]:
df_seems_relevant.head()[kept_columns].corr()

In [ ]:
plot_histogram(
    df_seems_relevant, col_name="topic_openalex", min_count_to_keep=5, height=900
)

In [ ]:
df_seems_relevant[df_seems_relevant["topic_dimensions"] == "C18 Law"]

In [ ]:
import pandas as pd
from typing import List
import numpy as np
from scipy.stats import chi2_contingency


def phi_coefficient(series1, series2):
    contingency_table = pd.crosstab(series1, series2)
    chi2, _, _, _ = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    phi = np.sqrt(chi2 / n)
    return phi


def ratio_coefficient(series1, series2):
    return (series1 == series2).sum() / len(series2)


def coefficiant_matrix(
    df_input: pd.DataFrame, col_names: List[str], coefficiant_func=phi_coefficient
):
    ret_matrix = pd.DataFrame(index=col_names, columns=col_names)

    for col1 in col_names:
        for col2 in col_names:
            ret_matrix.loc[col1, col2] = coefficiant_func(
                df_input[col1], df_input[col2]
            )
    return ret_matrix


# Example DataFrame
df = pd.DataFrame(
    {"A": [0, 1, 0, 1, 0, 1], "B": [0, 0, 1, 1, 0, 1], "C": [1, 1, 0, 0, 1, 0]}
)

# Compute phi correlation matrix

coefficiant_matrix(df, ["A", "B", "C"], coefficiant_func=ratio_coefficient)

In [ ]:
coef_matrix = coefficiant_matrix(df_analyisis, col_names=type_list_col.keys())
coef_matrix

In [ ]:
import plotly.graph_objects as go


def plot_correlation_matrix(corr_matrix, title="Correlation Matrix", zmid=0):
    """
    Plot a correlation matrix as a heatmap using Plotly.

    Parameters:
    - corr_matrix: pandas DataFrame or numpy array, the correlation matrix to plot
    - title: str, title of the plot (default: "Correlation Matrix")
    """
    # Create the figure
    fig = go.Figure(
        data=go.Heatmap(
            z=corr_matrix.values if hasattr(corr_matrix, "values") else corr_matrix,
            x=corr_matrix.columns
            if hasattr(corr_matrix, "columns")
            else np.arange(corr_matrix.shape[1]),
            y=corr_matrix.index
            if hasattr(corr_matrix, "index")
            else np.arange(corr_matrix.shape[0]),
            colorscale="RdBu",
            zmid=zmid,
            text=corr_matrix.values if hasattr(corr_matrix, "values") else corr_matrix,
            texttemplate="%{text:.2f}",
            textfont={"size": 10},
            colorbar=dict(title="Correlation"),
        )
    )

    # Update layout for better visualization
    fig.update_layout(
        title=title,
        xaxis=dict(side="bottom"),
        width=1200,
        height=800,
        autotypenumbers="convert types",
    )

    # Make the plot square and adjust margins
    fig.update_yaxes(scaleanchor="x")
    fig.update_layout(margin=dict(l=100, r=100, b=100, t=100))
    return fig


# Example usage:
# import pandas as pd

plot_correlation_matrix(coef_matrix)

In [ ]:
df_analyisis[df_analyisis["is_reference"]][kept_columns]

In [ ]:
plot_correlation_matrix(
    coefficiant_matrix(
        df_analyisis[df_analyisis["is_reference"]],
        col_names=type_list_col.keys(),
        coefficiant_func=ratio_coefficient,
    ),
    zmid=0.6,
)